In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/archive (9).zip"
extract_path = "/content/dataset"

if not os.path.exists("/content/dataset/socal2.csv"):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

print("Dataset ready ")

Dataset ready 


In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset/socal2.csv")
IMAGE_DIR = "/content/dataset/socal2/socal_pics/"

In [ ]:
df = df.dropna()
df = df[df["image_id"].notnull()]

In [ ]:
df["image_path"] = df["image_id"].astype(str).apply(
    lambda x: IMAGE_DIR + x + ".jpg"
)

In [ ]:
df = df.drop(columns=["street"])

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
df["citi"] = le_city.fit_transform(df["citi"])

le_ncity = LabelEncoder()
df["n_citi"] = le_ncity.fit_transform(df["n_citi"])

In [ ]:
import numpy as np

tab_cols = ["citi", "n_citi", "bed", "bath", "sqft"]

X_tab = df[tab_cols].values.astype(np.float32)
y = df["price"].values.astype(np.float32)

In [ ]:
from sklearn.model_selection import train_test_split

train_img, test_img, X_train, X_test, y_train, y_test = train_test_split(
    df["image_path"].values,
    X_tab,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img

In [ ]:
BATCH_SIZE = 8

def preprocess(img_path, tab, label):
    img = load_image(img_path)
    return (img, tab), label

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((
    train_img,
    X_train,
    y_train
))

train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices((
    test_img,
    X_test,
    y_test
))

test_ds = test_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
img_input = tf.keras.Input(shape=(224,224,3))

base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=img_input
)

base.trainable = False

x1 = tf.keras.layers.GlobalAveragePooling2D()(base.output)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
tab_input = tf.keras.Input(shape=(len(tab_cols),))

x2 = tf.keras.layers.Dense(128, activation="relu")(tab_input)
x2 = tf.keras.layers.Dense(64, activation="relu")(x2)

In [ ]:
x = tf.keras.layers.Concatenate()([x1, x2])

x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)

output = tf.keras.layers.Dense(1)(x)

In [ ]:
model = tf.keras.Model(inputs=[img_input, tab_input], outputs=output)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

Epoch 1/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 107s 48ms/step - loss: 125753253888.0000 - mae: 255265.4531 - val_loss: 98328821760.0000 - val_mae: 229907.4531
Epoch 2/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 84s 24ms/step - loss: 103150837760.0000 - mae: 234286.7188 - val_loss: 98187239424.0000 - val_mae: 230891.5625
Epoch 3/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 38s 24ms/step - loss: 102795296768.0000 - mae: 234678.4375 - val_loss: 98242076672.0000 - val_mae: 232022.4219
Epoch 4/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 44s 26ms/step - loss: 103331741696.0000 - mae: 234280.4375 - val_loss: 97895907328.0000 - val_mae: 228848.0000
Epoch 5/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 102576906240.0000 - mae: 233221.4844 - val_loss: 99187736576.0000 - val_mae: 227746.1562
Epoch 6/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 37s 23ms/step - loss: 103909244928.0000 - mae: 234198.5312 - val_loss: 98657026048.0000 - val_mae: 233618.5469
Epoch 7/10
1548/1548 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - loss: 103173005312.0000

In [ ]:
model.evaluate(test_ds)

387/387 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 98243018752.0000 - mae: 226937.0781


[98243018752.0, 226937.078125]

In [ ]:
def predict_price(image_path, citi, n_citi, bed, bath, sqft):
    img = load_image(image_path)
    img = tf.expand_dims(img, 0)

    tab = np.array([[citi, n_citi, bed, bath, sqft]], dtype=np.float32)

    return model.predict([img, tab])[0][0]

In [ ]:
predict_price(
    "/content/dataset/socal2/socal_pics/299.jpg",
    2, 3, 3, 2, 1500
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step


np.float32(410797.16)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
import numpy as np

y_pred = model.predict(test_ds).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

387/387 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step
MAE: 226937.03125
RMSE: 313437.38477724703
R2 Score: 0.33345484733581543


In [ ]:
model.save("/content/drive/MyDrive/house_model.keras")

In [ ]:
image_path = input("Enter image path: ").strip()
citi = int(input("City code: "))
n_citi = int(input("Neighborhood code: "))
bed = float(input("Beds: "))
bath = float(input("Baths: "))
sqft = float(input("Sqft: "))

price = predict_price(image_path, citi, n_citi, bed, bath, sqft)

print("Predicted Price:", price)

Enter image path: /content/dataset/socal2/socal_pics/299.jpg
City code: 1
Neighborhood code: 3
Beds: 3
Baths: 2
Sqft: 4900
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Predicted Price: 1244001.0


In [ ]:
predict_price(
    "/content/dataset/socal2/socal_pics/299.jpg",
    2, 3, 3, 2, 1500
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


np.float32(410797.16)

In [ ]:
model.save("/content/drive/MyDrive/house_model.keras")

In [ ]:
!pip install -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 whi

In [ ]:
import tensorflow as tf
import numpy as np
import gradio as gr


model = tf.keras.models.load_model(
    "/content/drive/MyDrive/house_model.keras"
)

IMG_SIZE = (224, 224)

def predict_ui(image, citi, n_citi, bed, bath, sqft):


    img = tf.image.resize(image, IMG_SIZE) / 255.0
    img = tf.expand_dims(img, 0)


    tab = np.array(
        [[citi, n_citi, bed, bath, sqft]],
        dtype=np.float32
    )


    pred = model.predict([img, tab])[0][0]

    return f"Predicted Price: {pred:.2f}"

interface = gr.Interface(
    fn=predict_ui,
    inputs=[
        gr.Image(type="numpy"),
        gr.Number(label="City Code"),
        gr.Number(label="Neighborhood Code"),
        gr.Number(label="Bedrooms"),
        gr.Number(label="Bathrooms"),
        gr.Number(label="Sqft")
    ],
    outputs="text",
    title="House Price Prediction AI",
    description="Upload image + enter house details to predict price"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a7e51862fb834b5c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
